# Careerjet job scraping

Try the Careerjet API first, then fall back to Selenium scraping.

## 1. Imports

In [5]:
import httpx
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import sqlite3, time

## 2. Try the API

In [6]:
url = "https://www.careerjet.it/offerte-lavoro"
params = {"s": "python", "l": "Italia"}
headers = {"User-Agent": "Mozilla/5.0"}
r = httpx.get(url, params=params, headers=headers)
print("status:", r.status_code)
print(r.text[:500])

status: 200
<!DOCTYPE html>
<html lang="it" dir="ltr">
<head>
<meta charset="utf-8">
<title>Careerjet</title>
<meta name="viewport" content="width=device-width, initial-scale=1, minimum-scale=1, maximum-scale=1, user-scalable=no, shrink-to-fit=no">
<style>
@keyframes svg_wait_rotation {
  0% {
    transform:translate3d(-50%, -50%, 0) rotate(0deg);
    //transform: rotate(0deg) translate3d(0, 0, 0);
  }
  100% {
    transform:translate3d(-50%, -50%, 0) rotate(360deg);
    //transform: rotate(360deg) translat


## 3. Fallback: scrape with Selenium

In [7]:
# Env settings needed on unix system with wayland issues

import os
os.environ['GDK_BACKEND']='x11'
os.environ['DISABLE_WAYLAND']='1'
os.environ['GTK_USE_PORTAL']='0'
os.environ['GSETTINGS_BACKEND']='memory'

In [8]:
options = webdriver.ChromeOptions()
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)
driver.get("https://www.careerjet.it/offerte-lavoro?s=python&l=Italia")
time.sleep(4)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)

job_cards = wait.until(EC.presence_of_all_elements_located((
    By.CSS_SELECTOR, "article, .job, .job-listing, [data-jobid], .listing"
)))

jobs = []
for card in job_cards:
    try:
        title_el = card.find_element(By.CSS_SELECTOR, "h2 a, h3 a, .title a, .job-title a, a")
        title = title_el.text.strip()
        link = title_el.get_attribute("href")
    except Exception:
        title, link = "", ""
    try:
        company = card.find_element(By.CSS_SELECTOR, ".company, .recruiter, [itemprop='hiringOrganization']").text.strip()
    except Exception:
        company = ""
    try:
        location = card.find_element(By.CSS_SELECTOR, ".location, [itemprop='jobLocation'], .place, .city").text.strip()
    except Exception:
        location = ""
    try:
        salary = card.find_element(By.CSS_SELECTOR, ".salary, .pay, [itemprop='baseSalary'], .wage, .compensation").text.strip()
    except Exception:
        salary = ""
    try:
        description = card.find_element(By.CSS_SELECTOR, ".description, .job-description, .summary, .snippet, p, [itemprop='description']").text.strip()
    except Exception:
        description = ""
    jobs.append({
        "title": title,
        "company": company,
        "location": location,
        "salary": salary,
        "description": description,
        "link": link,
    })

df = pd.DataFrame(jobs)
print(f"Found {len(df)} job postings")
print(df.head())


Found 20 job postings
                                            title          company  \
0                     Python Full Stack Developer  Energent S.p.A.   
1                     Python Full Stack Developer  Energent S.p.A.   
2                  PYTHON DEVELOPER - FULL REMOTE       LOG.IT srl   
3  Senior Full Stack Developer (Python / Angular)       Dedalo SRL   
4                        Python Software Engineer        DID s.r.l   

                       location                    salary      description  \
0                          Roma  30.000-36.000 € all'anno  Energent S.p.A.   
1                          Roma  25.000-30.000 € all'anno  Energent S.p.A.   
2                        Milano                                 LOG.IT srl   
3             Carinaro, Caserta                                 Dedalo SRL   
4  Provincia di Monza e Brianza                                  DID s.r.l   

                                                link  
0  https://www.careerjet.it/jobad